In [1]:
import pandas as pd 


In [2]:
df = pd.read_csv(r"D:\pypipeline\data\raw\static\hourly\aqi\baisipati_hourly.csv")

In [8]:
df.sort_values(by=['datetime'])

,station,particulate_matter,datetime,value
83523,Bhaisipati,TSP,2018-12-31T19:12:48Z,37.2
15332,Bhaisipati,PM1,2018-12-31T19:12:48Z,32.1
54645,Bhaisipati,PM2.5,2018-12-31T19:12:48Z,34.3
25976,Bhaisipati,PM10,2018-12-31T19:12:48Z,37.2
25977,Bhaisipati,PM10,2018-12-31T19:22:48Z,37.6
...,...,...,...,...
83521,Bhaisipati,TSP,2020-05-09T15:43:51Z,26.7
54644,Bhaisipati,PM2.5,2020-05-09T15:57:22Z,21.8
83522,Bhaisipati,TSP,2020-05-09T15:57:35Z,48.4
15331,Bhaisipati,PM1,2020-05-09T15:57:48Z,20.9


In [ ]:
df

In [3]:
df_pm25 = df[df["particulate_matter "]=="PM2.5"].drop(columns=["particulate_matter ","station"]).rename(columns={"value":"pm25_value"})
df_pm10 = df[df["particulate_matter "]=="PM10"].drop(columns=["particulate_matter ","station"]).rename(columns={"value":"pm10_value"})
df_pm1 = df[df["particulate_matter "]=="PM1"].drop(columns=["particulate_matter ","station"]).rename(columns={"value":"pm1_value"})
df_tsp = df[df["particulate_matter "]=="TSP"].drop(columns=["particulate_matter ","station"]).rename(columns={"value":"tsp_value"})  

In [4]:
df_merged = df_pm25.merge(df_pm10, on="datetime").merge(df_pm1, on="datetime").merge(df_tsp, on="datetime")

In [5]:
df_merged["datetime"] = pd.to_datetime(df_merged["datetime"], utc=True, infer_datetime_format=True)

C:\Users\ZENBOOK\AppData\Local\Temp\ipykernel_9544\3791337626.py:1: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df_merged["datetime"] = pd.to_datetime(df_merged["datetime"], utc=True, infer_datetime_format=True)


In [12]:
df_merged["datetime"] = df_merged["datetime"].dt.tz_convert("Asia/Kathmandu")

In [16]:
df_merged

,datetime,pm25_value,pm10_value,pm1_value,tsp_value
573,2019-01-01 00:57:48+05:45,34.3,37.2,32.1,37.2
574,2019-01-01 01:07:48+05:45,33.3,37.6,31.3,37.7
575,2019-01-01 01:27:48+05:45,52.2,75.6,35.3,132.2
576,2019-01-01 02:07:48+05:45,35.5,49.8,26.1,53.1
577,2019-01-01 02:57:48+05:45,30.6,34.3,29.5,37.1
...,...,...,...,...,...
453,2019-06-22 21:53:00+05:45,5.1,5.1,5.1,51.0
9987,2019-06-22 22:03:00+05:45,5.4,5.5,5.2,55.0
9988,2019-06-22 22:13:00+05:45,5.0,5.1,4.9,51.0
9989,2019-06-22 22:23:00+05:45,4.7,4.7,4.5,47.0


In [20]:
num_cols = ['pm25_value', 'pm10_value', 'pm1_value', 'tsp_value']
df_merged[num_cols] = df_merged[num_cols].apply(pd.to_numeric, errors='coerce')

df_hourly = df_merged.sort_values(by='datetime').resample('H', on='datetime').mean().reset_index().round(2)


C:\Users\ZENBOOK\AppData\Local\Temp\ipykernel_9544\910418770.py:4: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df_merged.sort_values(by='datetime').resample('H', on='datetime').mean().reset_index().round(2)


In [24]:
df_hourly

,pm25_value,pm10_value,pm1_value,tsp_value
datetime,,,,
2019-01-01 00:00:00+05:45,34.30,37.20,32.10,37.20
2019-01-01 01:00:00+05:45,42.75,56.60,33.30,84.95
2019-01-01 02:00:00+05:45,33.05,42.05,27.80,45.10
2019-01-01 03:00:00+05:45,32.50,38.30,29.10,38.90
2019-01-01 04:00:00+05:45,34.20,37.10,32.90,37.30
...,...,...,...,...
2019-06-22 18:00:00+05:45,1.70,1.70,1.64,17.00
2019-06-22 19:00:00+05:45,1.70,1.72,1.64,17.20
2019-06-22 20:00:00+05:45,3.97,3.98,3.85,39.83


In [22]:
df_hourly.set_index('datetime', inplace=True)

In [25]:
df_hourly.loc["2019-01-01"]

,pm25_value,pm10_value,pm1_value,tsp_value
datetime,,,,
2019-01-01 00:00:00+05:45,34.30,37.20,32.10,37.20
2019-01-01 01:00:00+05:45,42.75,56.60,33.30,84.95
2019-01-01 02:00:00+05:45,33.05,42.05,27.80,45.10
2019-01-01 03:00:00+05:45,32.50,38.30,29.10,38.90
2019-01-01 04:00:00+05:45,34.20,37.10,32.90,37.30
2019-01-01 05:00:00+05:45,33.35,37.05,31.20,37.05
2019-01-01 06:00:00+05:45,40.90,49.90,36.85,51.15
2019-01-01 07:00:00+05:45,44.90,51.65,41.05,52.05
2019-01-01 08:00:00+05:45,NaN,NaN,NaN,NaN
